In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import sqlite3
import pandas as pd

from src.analytics.ratios import (
    net_profit_margin,
    operating_profit_margin,
    return_on_equity,
    return_on_capital_employed,
    return_on_assets,
)

In [2]:
db_path = PROJECT_ROOT / "data" / "db" / "nifty100.db"

conn = sqlite3.connect(db_path)

print("Connected successfully.")

Connected successfully.


In [3]:
sample = pd.read_sql("""
SELECT
    p.company_id,
    p.year,
    p.sales,
    p.operating_profit,
    p.net_profit,
    p.interest,
    b.equity_capital,
    b.reserves,
    b.borrowings,
    b.total_assets
FROM profitandloss p
JOIN balancesheet b
ON p.company_id = b.company_id
AND p.year = b.year
WHERE p.company_id = 'ADANIPORTS'
LIMIT 1;
""", conn)

sample

,company_id,year,sales,operating_profit,net_profit,interest,equity_capital,reserves,borrowings,total_assets
0,ADANIPORTS,Mar 2013,3577.0,2382.0,1639.0,542.0,401.0,5993.0,11620.0,21035.0


In [4]:
row = sample.iloc[0]

print("Company :", row.company_id)
print("Year    :", row.year)

print("\nNet Profit Margin")
print(
    net_profit_margin(
        row.net_profit,
        row.sales
    )
)

print("\nOperating Profit Margin")
print(
    operating_profit_margin(
        row.operating_profit,
        row.sales
    )
)

print("\nROE")
print(
    return_on_equity(
        row.net_profit,
        row.equity_capital,
        row.reserves
    )
)

print("\nROCE")
print(
    return_on_capital_employed(
        row.operating_profit,
        row.interest,
        row.equity_capital,
        row.reserves,
        row.borrowings
    )
)

print("\nROA")
print(
    return_on_assets(
        row.net_profit,
        row.total_assets
    )
)

Company : ADANIPORTS
Year    : Mar 2013

Net Profit Margin
45.820519988817445

Operating Profit Margin
66.59211629857423

ROE
25.633406318423525

ROCE
16.23181969579216

ROA
7.791775612075114


In [5]:
profit = pd.read_sql(
    "SELECT * FROM profitandloss",
    conn
)

balance = pd.read_sql(
    "SELECT * FROM balancesheet",
    conn
)

ratio_df = profit.merge(
    balance,
    on=["company_id", "year"],
    how="inner",
    suffixes=("_pl", "_bs")
)

print(ratio_df.shape)

ratio_df.head()

(1151, 26)


,id_pl,company_id,year,sales,expenses,operating_profit,opm_percentage,other_income,interest,depreciation,...,equity_capital,reserves,borrowings,other_liabilities,total_liabilities,fixed_assets,cwip,investments,other_asset,total_assets
0,61,ABB,Dec 2012,1653.0,1451.0,202.0,12.0,33.0,0.0,19.0,...,21.0,626.0,0.0,260.0,907.0,109.0,1.0,0.0,798.0,907.0
1,62,ABB,Mar 2014,2276.0,2009.0,267.0,12.0,49.0,0.0,22.0,...,21.0,767.0,0.0,351.0,1139.0,98.0,1.0,0.0,1040.0,1139.0
2,63,ABB,Mar 2015,2289.0,1977.0,312.0,14.0,48.0,0.0,15.0,...,21.0,916.0,0.0,436.0,1374.0,96.0,4.0,0.0,1274.0,1374.0
3,64,ABB,Mar 2016,2614.0,2250.0,365.0,14.0,50.0,3.0,14.0,...,21.0,1174.0,0.0,421.0,1616.0,108.0,3.0,0.0,1505.0,1616.0
4,65,ABB,Mar 2017,2903.0,2505.0,398.0,14.0,57.0,2.0,16.0,...,21.0,1366.0,0.0,679.0,2066.0,110.0,6.0,0.0,1950.0,2066.0


In [6]:
ratio_df["net_profit_margin_pct"] = ratio_df.apply(
    lambda row: net_profit_margin(
        row["net_profit"],
        row["sales"]
    ),
    axis=1
)

ratio_df["operating_profit_margin_pct"] = ratio_df.apply(
    lambda row: operating_profit_margin(
        row["operating_profit"],
        row["sales"]
    ),
    axis=1
)

ratio_df["return_on_equity_pct"] = ratio_df.apply(
    lambda row: return_on_equity(
        row["net_profit"],
        row["equity_capital"],
        row["reserves"]
    ),
    axis=1
)

ratio_df["return_on_capital_employed_pct"] = ratio_df.apply(
    lambda row: return_on_capital_employed(
        row["operating_profit"],
        row["interest"],
        row["equity_capital"],
        row["reserves"],
        row["borrowings"]
    ),
    axis=1
)

ratio_df["return_on_assets_pct"] = ratio_df.apply(
    lambda row: return_on_assets(
        row["net_profit"],
        row["total_assets"]
    ),
    axis=1
)

print("Profitability ratios calculated successfully.")

Profitability ratios calculated successfully.


In [7]:
ratio_df[
    [
        "company_id",
        "year",
        "net_profit_margin_pct",
        "operating_profit_margin_pct",
        "return_on_equity_pct",
        "return_on_capital_employed_pct",
        "return_on_assets_pct"
    ]
].head(10)

,company_id,year,net_profit_margin_pct,operating_profit_margin_pct,return_on_equity_pct,return_on_capital_employed_pct,return_on_assets_pct
0,ABB,Dec 2012,8.771930,12.220206,22.411128,31.221020,15.986770
1,ABB,Mar 2014,8.699473,11.731107,25.126904,33.883249,17.383670
2,ABB,Mar 2015,10.004369,13.630406,24.439701,33.297759,16.666667
3,ABB,Mar 2016,9.755164,13.963275,21.338912,30.794979,15.779703
4,ABB,Mar 2017,9.541853,13.709955,19.971161,28.839221,13.407551
5,ABB,Mar 2018,12.158884,15.918739,23.685765,31.246308,16.597682
6,ABB,Mar 2019,12.231585,16.444686,22.410359,30.229084,15.300918
7,ABB,Mar 2020,14.488151,18.494991,24.393254,29.393707,16.718354
8,ABB,Mar 2021,16.032483,21.392111,26.556495,34.119782,17.994792
9,ABB,Mar 2022,16.262976,22.023204,28.333333,37.045760,18.915720
